# QAOA Vanilla - MaxCut on Weighted ER Graphs

This notebook:
- Generates 5 weighted Erdős–Rényi graphs
- Runs vanilla QAOA on each graph
- Collects expectation, variance, and optimal parameters

In [1]:
from qaoa import QAOA, problems, mixers, initialstates
from qiskit_algorithms.optimizers import L_BFGS_B
import time
from src.utils import generate_er_graphs, maxcut_bruteforce

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [2]:
# Configuration
n_graphs = 1
n_nodes = 10
depth = 10   # QAOA layers (p)

In [ ]:
# Generate graphs (already weighted from your function)
graphs = generate_er_graphs(n_graphs=n_graphs, n_nodes=n_nodes)

print(f"Generated {len(graphs)} graphs")

In [ ]:
# Inspect one graph
name, G = list(graphs.items())[0]
print("Example graph:", name)
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

In [ ]:

def graph_to_edgelist(G):
    """Convert NetworkX graph → [[u, v, w], ...]"""
    return [[u, v, d.get("weight", 1.0)] for u, v, d in G.edges(data=True)]


def run_qaoa_on_graph(graph_name, G, depth):
    print(f"\nRunning QAOA on {graph_name}")
    start = time.time()

    # --- Initialize QAOA
    qaoa = QAOA(
        problem=problems.MaxCut(G),
        mixer=mixers.X(),
        initialstate=initialstates.Plus(),
        interpolate=True,
        optimizer=[L_BFGS_B, {
            "maxiter": 50,
            "ftol": 1e-9
        }]
    )

    # --- Step 1: Landscape (optional, only meaningful for p=1)
    if depth == 1:
        qaoa.sample_cost_landscape()

    # --- Step 2: Optimization
    qaoa.optimize(depth=depth)

    # --- Step 3: Metrics
    exp_val = qaoa.get_Exp(depth=depth)
    var_val = qaoa.get_Var(depth=depth)

    gamma = qaoa.get_gamma(depth=depth)
    beta = qaoa.get_beta(depth=depth)

    # --- Step 4: Optimal (bruteforce)
    energy_opt, best_state = maxcut_bruteforce(G)

    # --- Step 5: Approximation ratio
    approx_ratio = exp_val / energy_opt if energy_opt != 0 else None

    # --- Step 6: Edge list
    edgelist_list = graph_to_edgelist(G)

    end = time.time()

    result = {
        "graph_name": graph_name,
        "n_nodes": G.number_of_nodes(),
        "edgelist_list_len": G.number_of_edges(),
        "n_layers": depth,
        "expected_energy": exp_val,
        "variance": var_val,
        "γ_coeff": gamma,
        "β_coeff": beta,
        "approx_ratio": approx_ratio,
        "energy_mqlib": energy_opt,
        "edgelist_list": edgelist_list,
        "took_time": round(end - start, 3),
        "method": "vanilla_qaoa",
        "optimizer": "BFGS"
    }

    print(f"Expectation: {exp_val}")
    print(f"Variance: {var_val}")

    return result

In [ ]:
results = []

for graph_name, G in graphs.items():
    result = run_qaoa_on_graph(graph_name, G, depth)
    results.append(result)

In [ ]:
# for x in dir(qaoa):
#     if not x.startswith("_"):    
#         print(x)

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(results)

df

In [ ]:
# Sort by best expectation (MaxCut → usually minimize energy)
df_sorted = df.sort_values(by="expected_energy")

df_sorted

In [ ]:
# Done
print("QAOA experiment complete ✅")

In [3]:
df = pd.read_csv("ADAPT.jl_results/test/9_nodes/qaoa_result/qaoa.csv")

In [7]:
len(df)

500

In [6]:
df.tail()

,graph_name,n_nodes,edgelist_list_len,n_layers,expected_energy,variance,γ_coeff,β_coeff,approx_ratio,energy_mqlib,edgelist_list,took_time,method,optimizer,run_id
495,graph_96,9,30,9,-10.092432,1.570227,[0.07940192 0.51459646 0.55928541 0.64787072 0.75313686 0.66667176\n 0.50271081 0.68890391 0.65063458],[1.6940925 1.88037318 1.69853409 1.95830331 2.12589586 2.24614794\n 2.10254056 1.81801036 1.89577252],0.864818,-11.67,"[[1, 2, 0.98], [1, 3, 0.17], [1, 4, 0.84], [1, 5, 0.13], [1, 7, 0.71], [1, 8, 0.15], [2, 3, 0.79], [2, 4, 0.63], [2, 5, 0.64], [2, 6, 0.85], [2, 8, 0.74], [2, 9, 0.92], [3, 4, 0.38], [3, 5, 0.46], [3, 6, 0.94], [3, 7, 0.42], [3, 8, 0.22], [3, 9, 0.59], [4, 5, 0.65], [4, 6, 0.38], [4, 7, 0.25], [4, 8, 0.57], [5, 6, 0.56], [5, 7, 0.73], [5, 9, 0.67], [6, 7, 0.78], [6, 8, 0.24], [6, 9, 0.38], [7, 9, 0.2], [8, 9, 0.57]]",60.760,vanilla_qaoa,BFGS,4
496,graph_97,9,25,9,-7.240117,1.494477,[0.94811359 1.01519412 0.97397838 0.98221111 0.95069546 0.94567597\n 0.97568536 1.00981491 1.01430506],[3.54563413 3.51133092 3.52968023 3.51839645 3.45329042 3.44585908\n 3.4426308 3.4366713 3.50112195],0.821807,-8.81,"[[1, 2, 0.23], [1, 4, 0.6], [1, 5, 0.75], [1, 6, 0.43], [1, 7, 0.32], [1, 8, 0.68], [2, 4, 0.56], [2, 5, 0.79], [2, 6, 0.99], [2, 8, 0.34], [2, 9, 0.81], [4, 7, 0.27], [4, 8, 0.24], [4, 9, 0.01], [5, 6, 0.31], [5, 8, 0.29], [5, 9, 0.22], [6, 3, 0.78], [6, 7, 0.47], [6, 8, 0.44], [3, 7, 0.52], [3, 8, 0.25], [3, 9, 0.05], [7, 8, 0.37], [7, 9, 0.67]]",47.030,vanilla_qaoa,BFGS,4
497,graph_98,9,21,9,-7.185107,0.866574,[0.52638053 0.50848074 0.60474593 0.58959954 0.55413777 0.53842938\n 0.53178154 0.6213171 0.5788387 ],[0.27965044 0.23460852 0.22985817 0.22448973 0.29633101 0.25411486\n 0.19031569 0.12777444 0.23223837],0.862558,-8.33,"[[1, 2, 0.35], [1, 3, 0.27], [1, 4, 0.5], [1, 7, 0.31], [2, 3, 0.65], [2, 5, 0.16], [2, 7, 0.65], [3, 6, 0.44], [3, 7, 0.93], [4, 5, 0.99], [4, 6, 0.51], [4, 7, 0.81], [4, 9, 0.63], [5, 6, 0.51], [5, 7, 0.98], [5, 8, 0.01], [5, 9, 0.6], [6, 7, 0.92], [6, 9, 0.05], [7, 9, 0.48], [8, 9, 0.91]]",47.770,vanilla_qaoa,BFGS,4
498,graph_99,9,28,9,-10.051992,1.723028,[0.51027524 0.54766826 0.56104584 0.56332657 0.56287525 0.56446904\n 0.57015174 0.57997623 0.59263567],[1.8497827 1.79153429 1.77617627 1.77824699 1.7843144 1.78866696\n 1.78924793 1.78383224 1.76644528],0.906401,-11.09,"[[1, 3, 0.78], [1, 4, 0.61], [1, 5, 0.51], [1, 7, 0.62], [1, 8, 0.74], [1, 9, 0.67], [3, 5, 0.43], [3, 6, 0.53], [3, 7, 0.58], [3, 8, 0.3], [3, 9, 0.96], [4, 5, 0.87], [4, 6, 0.53], [4, 7, 0.24], [4, 8, 0.83], [4, 9, 0.56], [5, 2, 0.88], [5, 7, 0.21], [5, 8, 0.78], [5, 9, 0.09], [2, 6, 0.61], [2, 7, 0.03], [2, 9, 0.26], [6, 7, 0.86], [6, 9, 0.26], [7, 8, 0.55], [7, 9, 0.1], [8, 9, 0.76]]",54.996,vanilla_qaoa,BFGS,4
499,graph_100,9,24,9,-7.758613,1.230461,[0.55224697 0.52293318 0.57385298 0.51658749 0.52016728 0.51425651\n 0.51008551 0.52636062 0.5169508 ],[0.31485745 0.20615039 0.29392947 0.24980137 0.34745248 0.34426041\n 0.21422938 0.23454337 0.28108743],0.836959,-9.27,"[[1, 2, 0.75], [1, 3, 0.55], [1, 4, 0.14], [1, 5, 0.95], [1, 6, 0.24], [1, 8, 0.46], [2, 4, 0.84], [2, 6, 0.64], [2, 8, 0.22], [2, 9, 0.28], [3, 4, 0.48], [3, 5, 0.28], [3, 6, 0.17], [3, 7, 0.28], [3, 9, 0.86], [4, 5, 0.66], [4, 6, 0.09], [4, 7, 0.18], [4, 8, 0.33], [5, 6, 0.82], [5, 8, 0.36], [5, 9, 0.45], [6, 9, 0.78], [8, 9, 0.75]]",50.969,vanilla_qaoa,BFGS,4
